### Experimen Pemprosesan data dan Pembersihan Data Transaksi Penjualan FMCG

#### Dibuat oleh : Damar Djati Wahyu Kemala
#### Juni 2026

>Tujuan dari pemrosesan data dan pembersihan data pada experimen ini adalah mengetahui berapa jumlah data kotor, dan bagaimana proses pembersihan terhadap data kotor itu ini untuk sample disusun oleh pembuat secara mandiri (bukan ambil sample dari dataset lain) (experimental)

>Data diambil dari Database pada Table staging.
>Secara real case di tempat kerja itu ada namanya staging yang datanya disimpan didalam temporary database. Setelah data dari staging tadi di bersihkan data tersebut baru masuk kedalam fact_table untuk dianalisis lebih lanjut.

## Copyright Personal Portfolio
* **Project Owner / Created By:** Damar Djati Wahyu Kemala
* **Role:** Aspiring Data Analyst & Business Analyst (Ex-SIMRS Developer)
* **Date Created:** Juni 2026
* **GitHub Portfolio:** [https://github.com/dams-code](https://github.com/dams-code)

---
*© 2026 Damar Djati Wahyu Kemala. This project is a part of my professional data analyst portfolio. Authorization is required for commercial use or modification.*

#### Proses Koneksi Ke SQL Server

In [3]:
import os
import pandas as pd
import numpy as np
import pyodbc
from dotenv import load_dotenv
from sklearn.linear_model import LinearRegression

load_dotenv(dotenv_path='../.env')

server = os.getenv("DB_SERVER", "localhost")
database = os.getenv("DB_NAME", "AnalitikFMCG_DB")
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

setKoneksi = f"DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={user};PWD={password}"

try:
    conn = pyodbc.connect(setKoneksi)
    print(f"koneksi ke SQL Server berhasil")
except Exception as e:
    print(f"Koneksi database SQL Server gagal: {e}")
    exit()

koneksi ke SQL Server berhasil


#### Load Data Dari Table Staging SQL Server

In [4]:
queryRawData = "SELECT * FROM Staging_Raw_Penjualan"
df = pd.read_sql(queryRawData, conn)

if df.empty:
    print("Database staging tidak ada data baru, proses diberhentikan");
    exit()
    
print(f"Berhasil memperoleh data kotor, {len(df)} data kotor")

Berhasil memperoleh data kotor, 100 data kotor


C:\Users\Damar\AppData\Local\Temp\ipykernel_15416\1071314300.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(queryRawData, conn)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   TransaksiID       100 non-null    str  
 1   Tanggal           94 non-null     str  
 2   Nama_Sales        99 non-null     str  
 3   Nama_Produk       99 non-null     str  
 4   Kuantitas         97 non-null     str  
 5   Harga_Satuan      94 non-null     str  
 6   Total_Pembayaran  91 non-null     str  
dtypes: str(7)
memory usage: 10.4 KB


#### Proses Cleaning Data

##### Konversi data yang bertipe varchar menjadi numeric

Tujuannya agar `Kuantitas`, `Harga_Satuan`, `Total_Pembayaran` dapat di hitung secara matematis dan dianalisa isi datanya.

In [6]:
df["Kuantitas"] = pd.to_numeric(df["Kuantitas"], errors='coerce')
df["Harga_Satuan"] = pd.to_numeric(df["Harga_Satuan"], errors='coerce')
df["Total_Pembayaran"] = pd.to_numeric(df["Total_Pembayaran"], errors='coerce')

In [7]:
df.head(6)

,TransaksiID,Tanggal,Nama_Sales,Nama_Produk,Kuantitas,Harga_Satuan,Total_Pembayaran
0,INV-2026001,2026-01-29,Budi,Kopi Kapal Api,15.0,NaN,NaN
1,INV-2026001,2026-01-29,Budi,Kopi Kapal Api,15.0,NaN,NaN
2,INV-2026001,2026-01-29,Budi,Kopi Kapal Api,15.0,NaN,NaN
3,INV-2026004,2026-02-20,Rani,AQUA 600ML,15.0,3000.0,45000.0
4,INV-2026005,2026-03-13,Budi,Teh Botol Sosro,28.0,5000.0,140000.0
5,INV-2026006,2026-02-09,Siti,Kopi Kapal Api,6.0,2000.0,12000.0


##### Membuang Data yang kosong

Data yang kosong yang dalam 1 barisnya isi pada kolom nama_produk dan total_pembayarannya kosong maka kita buang.

In [8]:
df.dropna(subset=["Nama_Produk", "Total_Pembayaran"], how='all', inplace=True)

##### Membuang Data Duplikat

In [9]:
df.drop_duplicates(subset=['TransaksiID'], keep='first', inplace=True)

##### Set standart pada Nama Produk

Nama Produk pada data ini dapat dibuat secara seragam dengan huruf besar.

In [10]:
df["Nama_Produk"] = df["Nama_Produk"].str.upper().str.strip()

##### Perbaikan Typo dan Miss-Format pada Data

In [11]:
mapping_produk = {
    'INDOMIE GORNG': 'INDOMIE GORENG',
    'INDOMIE GOR.': 'INDOMIE GORENG',
    'TEH BOTOL SRO': 'TEH BOTOL SOSRO',
    'KOPI KAPAL API 20G': 'KOPI KAPAL API'
}

df["Nama_Produk"] = df["Nama_Produk"].replace(mapping_produk)

In [12]:
df.head()

,TransaksiID,Tanggal,Nama_Sales,Nama_Produk,Kuantitas,Harga_Satuan,Total_Pembayaran
0,INV-2026001,2026-01-29,Budi,KOPI KAPAL API,15.0,NaN,NaN
3,INV-2026004,2026-02-20,Rani,AQUA 600ML,15.0,3000.0,45000.0
4,INV-2026005,2026-03-13,Budi,TEH BOTOL SOSRO,28.0,5000.0,140000.0
5,INV-2026006,2026-02-09,Siti,KOPI KAPAL API,6.0,2000.0,12000.0
6,INV-2026007,2026-04-02,Ahmad,CHITATO 68G,17.0,11000.0,187000.0


##### Perbaikan Format Tanggal yang Rusak

Set ke Awal Mei secara default jika terdapat format tanggal rusak,
kenapa awal mei supaya rentang tidak jauh dari populasi data yang banyaknya ditemui ada dibulan mei-juni 2026

In [13]:
df["Tanggal"] = pd.to_datetime(df["Tanggal"], errors='coerce', dayfirst='True')
df["Tanggal"] = df["Tanggal"].fillna(pd.Timestamp('2026-05-01'))

C:\Users\Damar\AppData\Local\Temp\ipykernel_15416\2546877184.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["Tanggal"] = pd.to_datetime(df["Tanggal"], errors='coerce', dayfirst='True')


##### Pada Data Terdapat Outlier

Perlu pembersihan outlier pada data (data yang menyimpang sangat jauh dari pola general pada kumpulan data). Pembersihan dengan cara pengambilan rentang `Kuantitas` yaitu mengambil rentang generalnya 0-500.

In [14]:
print(df[df['Kuantitas'] >= 500])

    TransaksiID    Tanggal Nama_Sales     Nama_Produk  Kuantitas  \
59  INV-2026060 2026-05-22      Ahmad  INDOMIE GORENG     5000.0   

    Harga_Satuan  Total_Pembayaran  
59        3500.0        17500000.0  


In [15]:
df = df[(df["Kuantitas"] > 0) & (df["Kuantitas"] <= 500)]

##### Kalkukasi Ulang perhitungan `Total_Pembayaran` dan isian satuan yang kosong.

In [16]:
df["Harga_Satuan"] = df["Harga_Satuan"].fillna(df["Total_Pembayaran"] / df["Kuantitas"])
df["Total_Pembayaran"] = df["Kuantitas"] * df["Harga_Satuan"]

In [17]:
print(f"Proses Cleaning selesai, Total {len(df)} data bersih")

Proses Cleaning selesai, Total 91 data bersih


##### Drop Harga satuan dan total pembayaran yang NaN

In [18]:
df.dropna(subset=["Harga_Satuan", "Total_Pembayaran"], how='all', inplace=True)

In [19]:
df.head()

,TransaksiID,Tanggal,Nama_Sales,Nama_Produk,Kuantitas,Harga_Satuan,Total_Pembayaran
3,INV-2026004,2026-02-20,Rani,AQUA 600ML,15.0,3000.0,45000.0
4,INV-2026005,2026-03-13,Budi,TEH BOTOL SOSRO,28.0,5000.0,140000.0
5,INV-2026006,2026-02-09,Siti,KOPI KAPAL API,6.0,2000.0,12000.0
6,INV-2026007,2026-04-02,Ahmad,CHITATO 68G,17.0,11000.0,187000.0
7,INV-2026008,2026-04-28,Rani,INDOMIE GORENG,6.0,3500.0,21000.0


##### Proses Machine Learning

Pada experimen kali ini saya menggunakan linear regression,
tujuannya untuk melakukan forecasting -> prediksi `Total_Pembayaran` berdasarkan `Kuantitas` dan `Harga_Satuan` produk.

Model Forecasting, saya menggunakan Linear Regression

> Target : Total_Pembayaran -> digunakan untuk prediksi. <br/>
> Fitur : Kuantitas, dan Harga_Satuan -> informasi sebagai bahan untuk prediksi target.

In [20]:
X_train = df[["Kuantitas", "Harga_Satuan"]]
y_train = df[["Total_Pembayaran"]]

In [21]:
model_experimen = LinearRegression()

In [22]:
model_experimen.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1, 2)","[[4383.47, 22.94]]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['Kuantitas','Harga_Satuan']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.","ndarray[float64](1,)",[-100925.25]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


##### Cek Skor R2-Score

In [23]:
print(f"Cek Skor pada R2-Score: {model_experimen.score(X_train, y_train)}")

Cek Skor pada R2-Score: 0.8247876631705577


##### Hasil skor Linear Regresi 82%

Linear Regression standartnya rentang skor ada pada 0.0 - 1, jika skor pada experimen yang saya buat ini mencapai 82 maka dilihat dari standartnya (akurasi baik).

> Akurasi baik pada train fit linear regression belum tentu menghasilkan predict yang bagus. (overfitting)

##### Prediksi Model

In [24]:
df["Prediksi_Omset_HariBerikutnya"] = model_experimen.predict(X_train) * 1.10

##### Hasil Predict Linear Regression

In [25]:
df.head()

,TransaksiID,Tanggal,Nama_Sales,Nama_Produk,Kuantitas,Harga_Satuan,Total_Pembayaran,Prediksi_Omset_HariBerikutnya
3,INV-2026004,2026-02-20,Rani,AQUA 600ML,15.0,3000.0,45000.0,37012.200089
4,INV-2026005,2026-03-13,Budi,TEH BOTOL SOSRO,28.0,5000.0,140000.0,150164.304781
5,INV-2026006,2026-02-09,Siti,KOPI KAPAL API,6.0,2000.0,12000.0,-31618.396465
6,INV-2026007,2026-04-02,Ahmad,CHITATO 68G,17.0,11000.0,187000.0,248529.734700
7,INV-2026008,2026-04-28,Rani,INDOMIE GORENG,6.0,3500.0,21000.0,6232.959643


## Copyright Personal Portfolio
* **Project Owner / Created By:** Damar Djati Wahyu Kemala
* **Role:** Aspiring Data Analyst & Business Analyst (Ex-SIMRS Developer)
* **Date Created:** Juni 2026
* **GitHub Portfolio:** [https://github.com/dams-code](https://github.com/dams-code)

---
*© 2026 Damar Djati Wahyu Kemala. This project is a part of my professional data analyst portfolio. Authorization is required for commercial use or modification.*